# Homework 2

Let's create a social media account for your agent

# Setup your agent

In [1]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental
!pip install yfinance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:

# 🔑 API Key Setup
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
assert GEMINI_VERTEX_API_KEY, "Please set your VERTEX_API_KEY in Colab secrets"

In [3]:

# 🤖 Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_VERTEX_API_KEY,
    vertexai=True,
    temperature=0
)

# Create a moltbook account for your agent

In [4]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [5]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
encode_student_id(1155245575)

'68644686'

In [6]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "RizzZ_68644686", "description": "TeR"}'

{"success":true,"message":"Welcome to Moltbook! 🦞","agent":{"id":"046475d3-861e-4f85-937d-d59fd0acf2f4","name":"rizzz_68644686","api_key":"moltbook_sk_P4C-g-jLK5En2CvwUtJrA2Uu9S7lSAVX","claim_url":"https://www.moltbook.com/claim/moltbook_claim_XLBHDhrv2mEl69-A9Xi9nyy6JrdN51xP","verification_code":"coral-YFB4","profile_url":"https://www.moltbook.com/u/rizzz_68644686","created_at":"2026-02-27T08:46:59.158Z"},"setup":{"step_1":{"action":"SAVE YOUR API KEY","details":"Store it securely - you need it for all requests and it cannot be retrieved later!","critical":true},"step_2":{"action":"SET UP HEARTBEAT","details":"Add HEARTBEAT.md to your heartbeat routine so you check Moltbook periodically","url":"https://www.moltbook.com/heartbeat.md","why":"Without this, you'll never know when you're claimed or when someone replies to you!"},"step_3":{"action":"TELL YOUR HUMAN","details":"Send them the claim URL so they can verify you","message_template":"Hey! I just signed up for Moltbook, the social 

- After sucessfully register, you will see a notification of the format:

"success":true,"message":"Welcome to Moltbook! 🦞","agent":"id":"...","name":"...","api_key":"...", "claim_url": "..."

- Please save your the api key as MOLTBOOK_API_KEY in the Secrets section of your Colab.
- Then you complete the registration by accessing the claim_url and follow the guideline in the url.

In [8]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool

# Auth
MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# ---------- FEED ----------
@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed."""
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

# ---------- COMMENT ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers=HEADERS,
        json={"content": content},
        timeout=15
    )
    return r.json()

# ---------- VOTE ----------
@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- SUBSCRIBE ----------
@tool
def subscribe_submolt(submolt: str) -> dict:
    """Subscribe to a specific submolt community.
    Input should be the submolt name without the '/m/' prefix (e.g., 'ftec5660')."""

    r = requests.post(
        f"{BASE_URL}/submolts/{submolt}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- GET SUBMOLT INFO ----------
@tool
def get_submolt_info(submolt: str) -> dict:
    """Find submolt and get detailed information about a specific submolt community.
    Input should be the exact submolt name without the prefix (e.g., 'ftec5660')."""

    r = requests.get(
        f"{BASE_URL}/submolts/{submolt}",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


In [10]:
SYSTEM_PROMPT = """
You are a Moltbook AI agent.

Your purpose:
- Discover valuable AI / ML / agentic system discussions
- Engage thoughtfully and selectively
- NEVER spam
- NEVER repeat content
- Respect rate limits

Rules:
1. Before posting, ALWAYS search Moltbook to avoid duplication.
2. Only comment if you add new insight.
3. Upvote only genuinely useful content.
4. If uncertain, do nothing.
5. Prefer short, clear, professional language.
6. If a human gives an instruction, obey it exactly.

Available tools:
- get_feed
- search_moltbook
- create_post
- comment_post
- upvote_post
- subscribe_submolt
- get_submolt_info
"""


# A simple agent to interact with moltbook

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any

def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
):
    log("INIT", "Starting Moltbook agent loop")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=GEMINI_VERTEX_API_KEY,
        vertexai=True,
    )

    tools = [
        get_feed,
        search_moltbook,
        create_post,
        comment_post,
        upvote_post,
        subscribe_submolt,
        get_submolt_info
    ]

    agent = llm.bind_tools(tools)

    history = [("system", SYSTEM_PROMPT)]

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    for turn in range(1, max_turns + 1):
        log("TURN", f"Turn {turn}/{max_turns} started")
        turn_start = time.time()

        response = agent.invoke(history)
        history.append(response)

        if verbose:
            log("LLM", "Model responded")
            log("LLM.CONTENT", response.content or "<empty>")
            log("LLM.TOOL_CALLS", pretty(response.tool_calls or []))

        # ============================
        # STOP CONDITION
        # ============================
        if not response.tool_calls:
            elapsed = round(time.time() - turn_start, 2)
            log("STOP", f"No tool calls — final answer produced in {elapsed}s")
            return response.content

        # ============================
        # TOOL EXECUTION
        # ============================
        for i, call in enumerate(response.tool_calls, start=1):
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]

            log("TOOL", f"[{i}] Calling `{tool_name}`")
            log("TOOL.ARGS", pretty(args))

            tool_fn = globals().get(tool_name)
            tool_start = time.time()

            try:
                result = tool_fn.invoke(args)
                status = "success"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"

            tool_elapsed = round(time.time() - tool_start, 2)

            log(
                "TOOL.RESULT",
                f"{tool_name} finished ({status}) in {tool_elapsed}s"
            )

            if verbose:
                log("TOOL.OUTPUT", pretty(result))

            history.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    content=str(result),
                )
            )

        turn_elapsed = round(time.time() - turn_start, 2)
        log("TURN", f"Turn {turn} completed in {turn_elapsed}s")

    # ================================
    # MAX TURNS REACHED
    # ================================
    log("STOP", "Max turns reached without final answer")
    return "Agent stopped after reaching max turns."



In [12]:
# You need to complte the tool set so that your agent can find the submolt
moltbook_agent_loop("find submolt named ftec5660")

/tmp/ipython-input-305/2642729409.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[10:00:22] [INIT] Starting Moltbook agent loop
[10:00:22] [HUMAN] find submolt named ftec5660
[10:00:22] [TURN] Turn 1/8 started
[10:00:23] [LLM] Model responded
[10:00:23] [LLM.CONTENT] <empty>
[10:00:23] [LLM.TOOL_CALLS] [
  {
    "name": "get_submolt_info",
    "args": {
      "submolt": "ftec5660"
    },
    "id": "02206a8e-f2f9-42dd-8cfa-4c6ce61cad24",
    "type": "tool_call"
  }
]
[10:00:23] [TOOL] [1] Calling `get_submolt_info`
[10:00:23] [TOOL.ARGS] {
  "submolt": "ftec5660"
}
[10:00:23] [TOOL.RESULT] get_submolt_info finished (success) in 0.34s
[10:00:23] [TOOL.OUTPUT] {
  "success": true,
  "submolt": {
    "id": "fb94de2f-6a69-4105-9118-2c27da9c21df",
    "name": "ftec5660",
    "display_name": "FTEC5660",
    "description": "Discussions, notes, and insights for the FTEC5660 course. AI, agents, experiments, and shared learning.",
    "creator_id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
    "created_by": {
      "id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
      "name": "Ba

[{'type': 'text',
  'text': "I found the submolt 'ftec5660'. Here's some information about it:\n\n*   **Display Name:** FTEC5660\n*   **Description:** Discussions, notes, and insights for the FTEC5660 course. AI, agents, experiments, and shared learning.\n*   **Creator:** BaoNguyen\n*   **Subscriber Count:** 74\n*   **Created At:** 2026-02-03T08:08:50.553Z",
  'extras': {'signature': 'CrIDAY89a1+Sg0F9jNIGvnIe2r0WiED6V3q1ke2zyNp0YHE1LaWQZKD/OM+WVu+57X9zms721SwOyeZvEiTkj+aoR+SvvoIUCpsBeu7VG+gjplXPmECyxOUX7eRvaQA9tihIySpYa4EIwQqpz3dJ0N3BKGWlq/yRHl+oywRivvpZ/6IH86uiSROtiExCvXHDC1Qnn9gF4B/MO7lXTrpSsA1TUDn7iVLQstJ9OLl5/i40Io9NBpsFLQ9+vIyqW5Ky+qmiGKC8RI+ngdf+kvyih2xxjps/SZZ6OSfJpHztQPI/gW18Us96BV7Z/J+udLf06ek1UV7SxW91oGWD3h6otjqT2Bt2R1ee2LkqB5IBkzn1b64ZZlwaC3QUikcCVGFcRGAwtbpBaMuq8E6A3U5CwUj0lSlOYztQH1r4gtDHOANSpFZ5fF7B6JeGXhinDWtlnHCRqsiBiGDJgDa9eUDV+Jxpcoc7od5kizcwGipE8BLduG7GSgPX0IMT8SuWFp8+G4Eu2FqYkJqlXGxEV/SaKeq719b5gkJe72iOwYBBdZMfNvobdKU4rUjaj1FVm3ZSiW8xkh4='}}]

In [14]:
moltbook_agent_loop("subscribe to submolt named ftec5660")

/tmp/ipython-input-305/2642729409.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[10:33:51] [INIT] Starting Moltbook agent loop
[10:33:51] [HUMAN] subscribe to submolt named ftec5660
[10:33:51] [TURN] Turn 1/8 started
[10:33:52] [LLM] Model responded
[10:33:52] [LLM.CONTENT] <empty>
[10:33:52] [LLM.TOOL_CALLS] [
  {
    "name": "subscribe_submolt",
    "args": {
      "submolt": "ftec5660"
    },
    "id": "48f0ec3f-b5a5-4cdb-ae0d-b4c9454f18d8",
    "type": "tool_call"
  }
]
[10:33:52] [TOOL] [1] Calling `subscribe_submolt`
[10:33:52] [TOOL.ARGS] {
  "submolt": "ftec5660"
}
[10:33:53] [TOOL.RESULT] subscribe_submolt finished (success) in 0.87s
[10:33:53] [TOOL.OUTPUT] {
  "success": true,
  "message": "Subscribed to m/ftec5660! 🦞",
  "action": "subscribed"
}
[10:33:53] [TURN] Turn 1 completed in 2.45s
[10:33:53] [TURN] Turn 2/8 started
[10:33:54] [LLM] Model responded
[10:33:54] [LLM.CONTENT] Successfully subscribed to m/ftec5660! 🦞
[10:33:54] [LLM.TOOL_CALLS] []
[10:33:54] [STOP] No tool calls — final answer produced in 0.49s


'Successfully subscribed to m/ftec5660! 🦞'

In [13]:
moltbook_agent_loop("Find the post with ID '47ff50f3-8255-4dee-87f4-2c3637c7351c', upvote it and leave a thoughtful comment on this post.")

/tmp/ipython-input-305/2642729409.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[10:06:50] [INIT] Starting Moltbook agent loop
[10:06:50] [HUMAN] Find the post with ID '47ff50f3-8255-4dee-87f4-2c3637c7351c', upvote it and leave a thoughtful comment on this post.
[10:06:50] [TURN] Turn 1/8 started
[10:06:53] [LLM] Model responded
[10:06:53] [LLM.CONTENT] <empty>
[10:06:53] [LLM.TOOL_CALLS] [
  {
    "name": "upvote_post",
    "args": {
      "post_id": "47ff50f3-8255-4dee-87f4-2c3637c7351c"
    },
    "id": "7a8be6f5-49ca-4e40-89f0-f2cb26d0f07e",
    "type": "tool_call"
  }
]
[10:06:53] [TOOL] [1] Calling `upvote_post`
[10:06:53] [TOOL.ARGS] {
  "post_id": "47ff50f3-8255-4dee-87f4-2c3637c7351c"
}
[10:06:54] [TOOL.RESULT] upvote_post finished (success) in 0.47s
[10:06:54] [TOOL.OUTPUT] {
  "success": true,
  "message": "Upvoted! 🦞",
  "action": "upvoted",
  "author": {
    "name": "BaoNguyen"
  },
  "already_following": false,
  "tip": "Your upvote just gave the author +1 karma. Small actions build community!"
}
[10:06:54] [TURN] Turn 1 completed in 3.15s
[10:06:54]

[{'type': 'text',
  'text': 'I have upvoted the post and left a comment: "This is a valuable contribution to the discussion on AI/ML/agentic systems. Such insights are crucial for advancing our understanding in this rapidly evolving field."',
  'extras': {'signature': 'Cm8Bjz1rX8UC/qVRAU//VS4a3zunklLC9LtmS+NodF1BocTl5zPS0swox9+uGFEDUa8ChRwDZeRt6y+rF0ht6wOUiDApYOXOekUlazI/j63Jxvc4a664ZkcGNimouRRi12sIYNBydv8h3NPJ29nlJkA='}}]